# Buyer Segmentation & Investment Profiling — EDA and Clustering

Machine Learning based Buyer Segmentation and Investment Profiling for Real Estate Market Intelligence
(Parcl Co. Limited × Unified Mentor)

This notebook walks through the same pipeline as `src/run_analysis.py`, interactively:
1. Load & clean `clients.csv` / `properties.csv`
2. Engineer client-level features
3. Explore the data (EDA)
4. Encode, scale, and cluster (K-Means + Hierarchical)
5. Select k via Elbow Method + Silhouette Score
6. Interpret & name the resulting buyer segments
7. Export the labelled dataset used by the Streamlit dashboard


In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline


## 1. Load & clean data

In [ ]:
from data_processing import get_processed_data

clients, properties, features = get_processed_data(data_dir="../data")
print("Clients:", clients.shape)
print("Properties:", properties.shape)
print("Client feature table:", features.shape)
features.head()


## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
clients["client_type"].value_counts().plot(kind="bar", ax=axes[0]); axes[0].set_title("Client Type")
clients["gender"].value_counts().plot(kind="bar", ax=axes[1]); axes[1].set_title("Gender")
clients["acquisition_purpose"].value_counts().plot(kind="bar", ax=axes[2]); axes[2].set_title("Acquisition Purpose")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9,4.5))
clients["country"].value_counts().plot(kind="bar")
plt.title("Clients by Country"); plt.ylabel("Number of clients")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(8,4.5))
sns.histplot(clients["age"], bins=30, kde=True)
plt.title("Client Age Distribution")
plt.tight_layout(); plt.show()


In [ ]:
corr_cols = ["age", "satisfaction_score", "loan_applied_flag", "num_purchases",
             "total_spend", "avg_purchase_price", "total_floor_area"]
plt.figure(figsize=(7,6))
sns.heatmap(features[corr_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout(); plt.show()


## 3. Encode, scale & choose k

In [ ]:
from clustering import prepare_matrix, evaluate_k_range

df, X, preprocessor = prepare_matrix(features)
eval_df = evaluate_k_range(X, k_range=range(2, 9))
eval_df


In [ ]:
fig, ax1 = plt.subplots(figsize=(8,5))
ax1.plot(eval_df["k"], eval_df["inertia"], marker="o", color="tab:blue")
ax1.set_xlabel("k"); ax1.set_ylabel("Inertia", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(eval_df["k"], eval_df["silhouette"], marker="s", color="tab:orange")
ax2.set_ylabel("Silhouette", color="tab:orange")
plt.title("Elbow Method & Silhouette Score")
plt.tight_layout(); plt.show()


## 4. Fit K-Means (k=4) and validate with Hierarchical clustering

In [ ]:
from clustering import run_full_pipeline

result = run_full_pipeline(features, k=4)
print("Segment names:", result["segment_names"])
print("Adjusted Rand Index (KMeans vs Hierarchical):", result["ari"])
result["summary"]


In [ ]:
seg_df = result["df"]
plt.figure(figsize=(8,6))
sns.scatterplot(x="pca_1", y="pca_2", hue="segment_name", data=seg_df, palette="deep", s=35, alpha=0.75)
plt.title("Client Segments (PCA-reduced feature space)")
plt.legend(bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout(); plt.show()


## 5. Segment profiling

In [ ]:
seg_df.groupby("segment_name")[["age","total_spend","avg_purchase_price",
                                 "loan_applied_flag","satisfaction_score"]].mean().round(2)


## 6. Export labelled dataset for the Streamlit dashboard

In [ ]:
seg_df.to_csv("../data/clustered_clients.csv", index=False)
summary = result["summary"].copy()
summary["segment_name"] = summary.index.map(result["segment_names"])
summary.to_csv("../data/cluster_summary.csv")
eval_df.to_csv("../data/k_evaluation.csv", index=False)
print("Saved.")
